In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import sys
if "../../../probability/spinoffs/autobnn" not in sys.path:
    sys.path.insert(0, "../../../probability/spinoffs/autobnn")
import autobnn
from autobnn import estimators, training_util
from autobnn import kernels, likelihoods

#!/usr/bin/env python3
import os, sys, time, logging
import numpy as np
import pandas as pd
import argparse
import matplotlib.pyplot as plt

# Import project modules
if "/mnt/research/ascsn/aaron-philip/barrier-extraction/src" not in sys.path:
    sys.path.append("/mnt/research/ascsn/aaron-philip/barrier-extraction/src")
from train import  init_E0, run_wong_trial, fit_wong_with_restarts, make_x_pred, draw_sigma_samples,percentile_bands,barrier_from_sigma_samples,fit_wong
from utils import load_packed_hdf5 # updated import
from wong import sigma

In [2]:
i = 623
data_path = "/mnt/home/philipaa/scratch/barrier-paper-datasets/cv0.75_spacing1.0.h5"
ds = load_packed_hdf5(data_path)
E_train, Sigma_train, Sigma_err_train, params_i = ds["get_i"](i) # note: uses init_E0 inside
E_pred = make_x_pred(E_train)

In [3]:
# 1) Initialize PRNG
seed = jax.random.PRNGKey(0)

# 3) Normalize inputs once
x_scale = np.std(E_train)
x_shift = np.mean(E_train)
y_scale = np.std(Sigma_train)
y_shift = np.mean(Sigma_train)

x_norm = ((E_train - x_shift) / x_scale)[..., None]
y_norm = ((Sigma_train-y_shift) / y_scale)[..., None]

# 4) Build & train the AutoBnnMapEstimator
seed, est_seed = jax.random.split(seed)

bnn_est = estimators.AutoBnnMCMCEstimator(
    'changepoint_of_safe_products',              
    likelihood_model='normal_likelihood_lognormal_noise',
    seed=est_seed,
    width=50,
    num_chains=4,
    num_draws=50,
    num_warmup=1000,
    distributed=1
)
"""
bnn_est = estimators.AutoBnnMapEstimator(
    rbf,#'sum_of_products',              
    likelihood_model='normal_likelihood_logistic_noise',
    seed=est_seed,
    width=50,
    num_iters= 5000,
    num_particles = 100,
)
"""
bnn_est = bnn_est.fit(x_norm, y_norm)

KeyboardInterrupt: 